# 03 · CV Pipeline — Segmentation Approach (v2) — self-contained

Same pipeline as `notebooks/03_seg_pipeline.ipynb` but **all `src` code is inlined**
below (no `import src`), so this notebook runs standalone on any machine (Windows
included). Insects are **instance-segmented** (mask + type); flowers are detected
(separate box per flower); visits are counted per flower / per insect type with
tracked IDs (debounced).

## Inlined source
Each cell below defines one `src` module as an in-notebook module object. Edit here.

In [ ]:
import os, sys, json, glob
from pathlib import Path
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
try:
    import cv2, torch
except Exception:
    pass
_REPO = Path.cwd()
for _c in [_REPO, *_REPO.parents]:
    if (_c / "data").exists() and (_c / "src").exists(): _REPO = _c; break
os.chdir(_REPO)
# ---- src/config.py (inlined; edit here) ----
"""Central configuration for the Bee-A-Hero data pipeline.

All paths are anchored to the repository root (this file lives in ``src/``),
so the pipeline is portable across machines and does not depend on any
absolute/Windows path. Import this module rather than hard-coding paths.
"""

from pathlib import Path

# --- Repository layout -------------------------------------------------------
REPO_ROOT: Path = _REPO

DATA_DIR: Path = REPO_ROOT / "data"
RAW_DIR: Path = DATA_DIR / "raw"
INTERIM_DIR: Path = DATA_DIR / "interim"
PROCESSED_DIR: Path = DATA_DIR / "processed"
BACKUP_DIR: Path = DATA_DIR / "_backup"

INAT_DIR: Path = RAW_DIR / "iNaturist"
FLOWER_DIR: Path = RAW_DIR / "Flower"
# Roboflow bee-detection COCO export (BEE.v8i). Place it here on any machine
# (git-ignored); overridable via the BEE_COCO_DIR env var for other layouts.
BEE_COCO_DIR: Path = Path(__import__("os").environ.get("BEE_COCO_DIR", RAW_DIR / "BEE_coco"))
# Videos to run the visit-counter on.
TEST_VIDEO_DIR: Path = RAW_DIR / "Test_Video"

# Published trained-weights repo on the Hugging Face Hub (for teammates to pull).
HF_WEIGHTS_REPO: str = "Manheim/bee-a-hero-cv"

# iNaturalist splits present on disk. ``public_test`` is unlabeled
# (annotations: 0) and is inference-only — it is NOT part of the labeled split.
INAT_LABELED_SPLITS: tuple[str, ...] = ("train_mini", "val")
INAT_UNLABELED_SPLIT: str = "public_test"

# Generated-artifact locations (git-ignored; see .gitignore data/interim/*).
MANIFEST_DIR: Path = INTERIM_DIR / "manifests"
EDA_DIR: Path = INTERIM_DIR / "eda"
REMOVED_DIR: Path = BACKUP_DIR / "removed"

# --- Reproducibility ---------------------------------------------------------
SEED: int = 42

# --- Split configuration -----------------------------------------------------
# Train/val/test proportions carved from the pooled labeled Insecta images.
SPLIT_RATIOS: dict[str, float] = {"train": 0.70, "val": 0.15, "test": 0.15}

# --- Taxonomy targets --------------------------------------------------------
# Keep only folders whose taxonomic Class == Insecta.
TARGET_CLASS: str = "Insecta"

# Bee families (Hymenoptera) tagged as a subset of interest for the project.
BEE_FAMILIES: frozenset[str] = frozenset({
    "Andrenidae", "Apidae", "Colletidae", "Halictidae",
    "Megachilidae", "Melittidae", "Stenotritidae",
})

# Valid image extensions.
IMAGE_EXTS: frozenset[str] = frozenset({".jpg", ".jpeg", ".png"})

from types import SimpleNamespace
C = SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper()})
REPO = _REPO
RUNS = C.INTERIM_DIR / "cv_runs"
def exists(p): return Path(p).exists()
print("repo:", REPO, "| device:", "cuda" if ("torch" in dir() and torch.cuda.is_available()) else "cpu")

#### `src/data_pipeline/flower/make_detection_dataset.py` (inlined)

In [ ]:
# ===== src/data_pipeline/flower/make_detection_dataset.py  (inlined module — edit here) =====
import types as _t, sys as _s
mdd = _t.ModuleType("mdd")
mdd.C = C
mdd.__file__ = str(_REPO / "src" / "data_pipeline/flower/make_detection_dataset.py")
_src = r'''
"""
make_detection_dataset.py
-------------------------
Turn the merged Flower Classification dataset (class subfolders) into an
OBJECT-DETECTION-ready dataset in YOLO format, AND emit a labels.csv that
maps every image to its class.

IMPORTANT / HONEST NOTE
-----------------------
The source is a *classification* dataset: one centered flower per image, with
NO ground-truth boxes. This script GENERATES boxes automatically with OpenCV
GrabCut foreground segmentation (a tight box around the detected flower region).
These boxes are approximate and machine-made, not human-verified. They give you
"clear boundaries" to train/prototype a detector, but you should spot-check and
hand-correct a sample before trusting them as ground truth.

INPUT  (created by merge_flowers.py):
    merged_dataset/
        Training Data/<Class>/*.jpeg
        Validation Data/<Class>/*.jpeg
        Testing Data/<Class>/*.jpeg

OUTPUT:
    yolo_dataset/
        images/{train,val,test}/<Class>_<orig>.jpeg
        labels/{train,val,test}/<Class>_<orig>.txt     # "<cls> cx cy w h" (normalized)
        data.yaml
        classes.txt
    labels.csv         # image -> class mapping (classification)
    annotations.csv    # image -> class + pixel bbox (detection, human-readable)

Usage:
    python make_detection_dataset.py
    python make_detection_dataset.py --src merged_dataset --out yolo_dataset --workers 16
    python make_detection_dataset.py --limit 20    # quick smoke test (20 imgs/class)
"""

import argparse
import csv
import os
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np

IMAGE_EXTS = {".jpeg", ".jpg", ".png"}
SPLIT_MAP = {
    "Training Data": "train",
    "Validation Data": "val",
    "Testing Data": "test",
}
GC_SIZE = 160          # work resolution for GrabCut (speed)
GC_ITERS = 3           # GrabCut iterations
MIN_AREA_FRAC = 0.02   # if fg smaller than this -> fallback box
MAX_AREA_FRAC = 0.985  # if fg bigger than this  -> fallback box
FALLBACK_FRAC = 0.92   # centered fallback box size (fraction of image)


def auto_bbox(img):
    """Return (cx,cy,w,h) normalized [0,1] for the main flower region.
    Falls back to a centered box if segmentation is unreliable."""
    h, w = img.shape[:2]
    scale = GC_SIZE / max(h, w)
    sw, sh = max(1, int(w * scale)), max(1, int(h * scale))
    small = cv2.resize(img, (sw, sh))

    mask = np.zeros((sh, sw), np.uint8)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    m = 0.06  # border margin -> assumed background
    rect = (int(sw * m), int(sh * m), int(sw * (1 - 2 * m)), int(sh * (1 - 2 * m)))

    used_fallback = False
    try:
        cv2.grabCut(small, mask, rect, bgd, fgd, GC_ITERS, cv2.GC_INIT_WITH_RECT)
        fg = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype("uint8")
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
        cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            x, y, bw, bh = cv2.boundingRect(c)
            area_frac = (bw * bh) / float(sw * sh)
            if not (MIN_AREA_FRAC <= area_frac <= MAX_AREA_FRAC):
                used_fallback = True
        else:
            used_fallback = True
    except Exception:
        used_fallback = True

    if used_fallback:
        f = FALLBACK_FRAC
        cx, cy, nw, nh = 0.5, 0.5, f, f
    else:
        cx = (x + bw / 2) / sw
        cy = (y + bh / 2) / sh
        nw = bw / sw
        nh = bh / sh

    # clamp
    cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
    nw, nh = min(nw, 1), min(nh, 1)
    return cx, cy, nw, nh, used_fallback


def process_one(task):
    """Worker: compute bbox for a single image. Returns dict or None on read error."""
    src, split, cls, cls_id, dst_name = task
    img = cv2.imread(src)
    if img is None:
        return None
    h, w = img.shape[:2]
    cx, cy, nw, nh, fb = auto_bbox(img)
    # pixel coords for the human-readable annotations.csv
    bw_px, bh_px = nw * w, nh * h
    xmin = int(round(cx * w - bw_px / 2))
    ymin = int(round(cy * h - bh_px / 2))
    xmax = int(round(cx * w + bw_px / 2))
    ymax = int(round(cy * h + bh_px / 2))
    xmin, ymin = max(0, xmin), max(0, ymin)
    xmax, ymax = min(w, xmax), min(h, ymax)
    return {
        "src": src, "split": split, "cls": cls, "cls_id": cls_id,
        "dst_name": dst_name, "w": w, "h": h,
        "cx": cx, "cy": cy, "nw": nw, "nh": nh,
        "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax,
        "fallback": fb,
    }


def build_tasks(src_root, limit):
    classes = set()
    tasks = []
    per_class_counter = {}
    for split_dir, split_key in SPLIT_MAP.items():
        sdir = src_root / split_dir
        if not sdir.is_dir():
            continue
        for cls_dir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            cls = cls_dir.name
            classes.add(cls)
    class_list = sorted(classes)
    class_id = {c: i for i, c in enumerate(class_list)}

    seen_names = set()
    for split_dir, split_key in SPLIT_MAP.items():
        sdir = src_root / split_dir
        if not sdir.is_dir():
            continue
        for cls_dir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            cls = cls_dir.name
            key = (split_key, cls)
            per_class_counter.setdefault(key, 0)
            for f in sorted(cls_dir.iterdir()):
                if f.suffix.lower() not in IMAGE_EXTS:
                    continue
                if limit and per_class_counter[key] >= limit:
                    break
                per_class_counter[key] += 1
                # unique, collision-proof destination name (flat per split).
                # Dedup on the STEM (not full name) so that e.g. "X.jpeg" and
                # "X.png" don't collide onto the same label .txt file.
                base = f"{cls}_{f.name}"
                stem = Path(base).stem
                dst = base
                i = 1
                while (split_key, Path(dst).stem) in seen_names:
                    dst = f"{stem}_{i}{f.suffix}"
                    i += 1
                seen_names.add((split_key, Path(dst).stem))
                tasks.append((str(f), split_key, cls, class_id[cls], dst))
    return tasks, class_list, class_id


# repo root = .../Bee-A-Hero  (this file is at src/data_pipeline/flower/make_detection_dataset.py)
REPO = Path(__file__).resolve().parents[3]
DEFAULT_SRC = REPO / "data" / "processed" / "flower" / "classification"
DEFAULT_OUT = REPO / "data" / "processed" / "flower" / "yolo"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default=str(DEFAULT_SRC),
                    help="classification dataset (default: data/processed/flower/classification)")
    ap.add_argument("--out", default=str(DEFAULT_OUT),
                    help="YOLO output (default: data/processed/flower/yolo)")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 4) - 2))
    ap.add_argument("--limit", type=int, default=0, help="max images per class+split (0 = all)")
    args = ap.parse_args()

    src_root = Path(args.src)
    out = Path(args.out)
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    tasks, class_list, class_id = build_tasks(src_root, args.limit)
    print(f"Images to process: {len(tasks)}  |  classes: {len(class_list)}  |  workers: {args.workers}")

    labels_rows = []       # image_path, split, class, class_id
    ann_rows = []          # image_path, width, height, class, class_id, xmin, ymin, xmax, ymax
    fallback_count = 0
    done = 0

    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(process_one, t) for t in tasks]
        for fut in as_completed(futs):
            r = fut.result()
            done += 1
            if r is None:
                continue
            if r["fallback"]:
                fallback_count += 1

            split = r["split"]
            dst_img = out / "images" / split / r["dst_name"]
            dst_lbl = out / "labels" / split / (Path(r["dst_name"]).stem + ".txt")

            shutil.copy2(r["src"], dst_img)
            with open(dst_lbl, "w") as fh:
                fh.write(f"{r['cls_id']} {r['cx']:.6f} {r['cy']:.6f} {r['nw']:.6f} {r['nh']:.6f}\n")

            rel = f"images/{split}/{r['dst_name']}"
            labels_rows.append([rel, split, r["cls"], r["cls_id"]])
            ann_rows.append([rel, r["w"], r["h"], r["cls"], r["cls_id"],
                             r["xmin"], r["ymin"], r["xmax"], r["ymax"]])

            if done % 2000 == 0:
                print(f"  ...{done}/{len(tasks)}")

    # sort for stable output
    labels_rows.sort()
    ann_rows.sort()

    with open(out / "labels.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["image", "split", "class", "class_id"])
        w.writerows(labels_rows)

    with open(out / "annotations.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["image", "width", "height", "class", "class_id",
                    "xmin", "ymin", "xmax", "ymax"])
        w.writerows(ann_rows)

    with open(out / "classes.txt", "w") as fh:
        fh.write("\n".join(class_list) + "\n")

    with open(out / "data.yaml", "w") as fh:
        fh.write(f"path: {out.resolve()}\n")
        fh.write("train: images/train\n")
        fh.write("val: images/val\n")
        fh.write("test: images/test\n")
        fh.write(f"nc: {len(class_list)}\n")
        fh.write("names:\n")
        for c in class_list:
            fh.write(f"  - {c}\n")

    print("\n" + "=" * 60)
    print("DETECTION DATASET READY ->", out.resolve())
    print("=" * 60)
    print(f"images written : {len(labels_rows)}")
    print(f"fallback boxes : {fallback_count} "
          f"({100*fallback_count/max(1,len(labels_rows)):.1f}% used centered box)")
    print(f"classes        : {class_list}")
    print("files          : data.yaml, classes.txt, labels.csv, annotations.csv")


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/data_pipeline/flower/make_detection_dataset.py", "exec"), mdd.__dict__)
_s.modules["mdd"] = mdd
globals()["mdd"] = mdd

#### `src/cv_engine/prepare_insect.py` (inlined)

In [ ]:
# ===== src/cv_engine/prepare_insect.py  (inlined module — edit here) =====
import types as _t, sys as _s
prepare_insect = _t.ModuleType("prepare_insect")
prepare_insect.C = C
prepare_insect.__file__ = str(_REPO / "src" / "cv_engine/prepare_insect.py")
prepare_insect.auto_bbox = mdd.auto_bbox
_src = r'''
"""Build a 2-class insect YOLO detection dataset: pollinator vs non_pollinator.

iNaturalist is a *classification* set (no boxes), so boxes are generated with
GrabCut (reusing ``auto_bbox``) on the centered organism. Classes come from the
Phase-4 manifest ``is_bee`` flag (bee families = pollinator). Non-pollinator is
heavily over-represented (147k vs 3.7k), so it is sub-sampled per split (seeded)
to balance 1:1 with pollinator.

Real bee boxes from the Roboflow COCO sets on the Desktop (iNat-sourced +
video-frame bees) are converted to YOLO and added to the **pollinator** class to
improve mAP and robustness on real video.

Classes: 0 = pollinator, 1 = non_pollinator.

Output (git-ignored): data/interim/insect_det/{images,labels}/{train,val,test}
                      + data.yaml

CLI:  python -m src.cv_engine.prepare_insect
"""

import csv
import json
import multiprocessing
import random
import shutil
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2


try:
    _MP = multiprocessing.get_context("fork")   # Linux/Mac
except ValueError:
    _MP = multiprocessing.get_context()          # Windows: default (spawn)
OUT = C.INTERIM_DIR / "insect_det"
NAMES = ["pollinator", "non_pollinator"]

# Real-box bee detection set (Roboflow COCO export).
# Only BEE.v8i (single iNat-sourced garden bees) — the Honey Bee hive export
# (~16 tiny dense bees/frame, hive-monitoring domain) tanks detection mAP and
# does not match the garden flower-visit videos, so it is excluded from detection.
# Portable location (git-ignored): data/raw/BEE_coco/ (see src/config.BEE_COCO_DIR).
# If absent, the pipeline still runs on iNaturalist alone (roboflow step skipped).
ROBOFLOW_SETS = [
    C.BEE_COCO_DIR,
]
_RF_SPLIT = {"train": "train", "valid": "val", "test": "test"}


# --------------------------------------------------------------------------- #
# iNaturalist -> GrabCut boxes, 2 classes, balanced
# --------------------------------------------------------------------------- #
def _select_inat_records() -> list[tuple[str, str, int, str]]:
    """Return (abs_path, split, class_id, dst_name); non_pollinator balanced 1:1."""
    rows = list(csv.DictReader(open(C.MANIFEST_DIR / "split_manifest.csv")))
    by_split_cls: dict[tuple[str, int], list[dict]] = defaultdict(list)
    for r in rows:
        cls = 0 if r["is_bee"] == "1" else 1
        by_split_cls[(r["split"], cls)].append(r)

    rng = random.Random(C.SEED)
    out = []
    for split in ("train", "val", "test"):
        poll = by_split_cls[(split, 0)]
        non = by_split_cls[(split, 1)]
        # balance: keep all pollinators, sample equal non-pollinators
        non = sorted(non, key=lambda r: r["path"])
        rng.shuffle(non)
        non = non[: len(poll)]
        for cls, recs in ((0, poll), (1, non)):
            for r in recs:
                p = C.REPO_ROOT / r["path"]
                dst = f"inat_{Path(r['path']).stem}.jpg"
                out.append((str(p), split, cls, dst))
    return out


def _process_inat(task):
    src, split, cls, dst = task
    img = cv2.imread(src)
    if img is None:
        return (split, "read_error")
    cx, cy, nw, nh, fb = auto_bbox(img)
    shutil.copy2(src, OUT / "images" / split / dst)
    (OUT / "labels" / split / f"{Path(dst).stem}.txt").write_text(
        f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
    return (split, f"cls{cls}")


# --------------------------------------------------------------------------- #
# Roboflow COCO -> YOLO (all boxes -> class 0 pollinator)
# --------------------------------------------------------------------------- #
def _convert_roboflow() -> Counter:
    stats: Counter = Counter()
    for ds in ROBOFLOW_SETS:
        if not ds.is_dir():
            continue
        tag = ds.name.split(".")[0].replace(" ", "_")[:20]
        for rf_split, our_split in _RF_SPLIT.items():
            ann = ds / rf_split / "_annotations.coco.json"
            if not ann.exists():
                continue
            coco = json.load(open(ann))
            img_by_id = {im["id"]: im for im in coco["images"]}
            boxes_by_img: dict[int, list] = defaultdict(list)
            for a in coco["annotations"]:
                boxes_by_img[a["image_id"]].append(a["bbox"])  # [x,y,w,h] pixels
            for iid, im in img_by_id.items():
                src = ds / rf_split / im["file_name"]
                if not src.exists():
                    continue
                W, H = im["width"], im["height"]
                lines = []
                for x, y, w, h in boxes_by_img.get(iid, []):
                    cx, cy = (x + w / 2) / W, (y + h / 2) / H
                    lines.append(f"0 {cx:.6f} {cy:.6f} {w / W:.6f} {h / H:.6f}")
                if not lines:                      # skip images with no bee box
                    continue
                dst = f"rf_{tag}_{our_split}_{iid}.jpg"
                shutil.copy2(src, OUT / "images" / our_split / dst)
                (OUT / "labels" / our_split / f"{Path(dst).stem}.txt").write_text(
                    "\n".join(lines) + "\n")
                stats[f"{our_split}_roboflow"] += 1
    return stats


def export_single_class(dst: Path | None = None) -> str:
    """Derive a single-class ``insect`` detection set from the 2-class boxes.

    Reuses the GrabCut/Roboflow boxes already in ``insect_det`` (no re-compute):
    images are symlinked, labels rewritten to class 0. Used to train the
    high-mAP detector; the 2-class set stays for building classifier crops.
    """
    dst = dst or (C.INTERIM_DIR / "insect_det1")
    for split in ("train", "val", "test"):
        (dst / "images" / split).mkdir(parents=True, exist_ok=True)
        (dst / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img in (OUT / "images" / split).iterdir():
            link = dst / "images" / split / img.name
            if not link.exists():
                link.symlink_to(img.resolve())
        for lbl in (OUT / "labels" / split).glob("*.txt"):
            lines = ["0 " + " ".join(l.split()[1:])
                     for l in lbl.read_text().splitlines() if l.strip()]
            (dst / "labels" / split / lbl.name).write_text("\n".join(lines) + "\n")
    (dst / "data.yaml").write_text(
        f"# single-class insect detector (high-mAP stage 1)\n"
        f"path: {dst.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        f"nc: 1\nnames: [insect]\n")
    return str(dst / "data.yaml")


def export_classifier_crops(dst: Path | None = None, min_size: int = 20) -> dict:
    """Crop every labeled box from the 2-class set into an ImageFolder for the
    pollinator/non_pollinator classifier: ``insect_cls/<split>/<class>/<crop>.jpg``.
    """
    dst = dst or (C.INTERIM_DIR / "insect_cls")
    names = {0: "pollinator", 1: "non_pollinator"}
    counts: Counter = Counter()
    for split in ("train", "val", "test"):
        for cn in names.values():
            (dst / split / cn).mkdir(parents=True, exist_ok=True)
        for lbl in sorted((OUT / "labels" / split).glob("*.txt")):
            cand = list((OUT / "images" / split).glob(lbl.stem + ".*"))
            if not cand:
                continue
            img = cv2.imread(str(cand[0]))
            if img is None:
                continue
            H, W = img.shape[:2]
            for i, line in enumerate(lbl.read_text().splitlines()):
                p = line.split()
                if len(p) < 5:
                    continue
                c = int(p[0]); cx, cy, w, h = map(float, p[1:5])
                x1, y1 = max(0, int((cx - w / 2) * W)), max(0, int((cy - h / 2) * H))
                x2, y2 = min(W, int((cx + w / 2) * W)), min(H, int((cy + h / 2) * H))
                if x2 - x1 < min_size or y2 - y1 < min_size:
                    continue
                cv2.imwrite(str(dst / split / names[c] / f"{lbl.stem}_{i}.jpg"),
                            img[y1:y2, x1:x2])
                counts[f"{split}_{names[c]}"] += 1
    return dict(counts)


def _write_yaml() -> None:
    (OUT / "data.yaml").write_text(
        f"# 2-class insect detector: pollinator vs non_pollinator\n"
        f"path: {OUT.resolve()}\n"
        f"train: images/train\nval: images/val\ntest: images/test\n"
        f"nc: 2\nnames: [pollinator, non_pollinator]\n")


def run(workers: int | None = None) -> dict:
    for split in ("train", "val", "test"):
        (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

    tasks = _select_inat_records()
    stats: Counter = Counter()
    with ProcessPoolExecutor(max_workers=workers, mp_context=_MP) as ex:
        for split, kind in ex.map(_process_inat, tasks, chunksize=32):
            stats[f"inat_{split}_{kind}"] += 1
    stats.update(_convert_roboflow())
    _write_yaml()
    return {"inat_images": len(tasks), "counts": dict(stats),
            "data_yaml": str(OUT / "data.yaml")}


if __name__ == "__main__":
    import os
    print(json.dumps(run(workers=os.cpu_count()), indent=2))

'''
exec(compile(_src, "src/cv_engine/prepare_insect.py", "exec"), prepare_insect.__dict__)
_s.modules["prepare_insect"] = prepare_insect
globals()["prepare_insect"] = prepare_insect

#### `src/cv_engine/sam_to_seg.py` (inlined)

In [ ]:
# ===== src/cv_engine/sam_to_seg.py  (inlined module — edit here) =====
import types as _t, sys as _s
sam_to_seg = _t.ModuleType("sam_to_seg")
sam_to_seg.C = C
sam_to_seg.__file__ = str(_REPO / "src" / "cv_engine/sam_to_seg.py")
_src = r'''
"""Bootstrap a YOLO instance-segmentation dataset for insects using **box-prompted** SAM.

Why box prompts (not a center point): a center-point prompt makes SAM grab the most
salient blob at the image centre — on real flower-visit video that blob is often the
*flower*, so the earlier model learned to segment flowers. Instead we prompt SAM with
the **existing YOLO detection boxes** (real garden-bee scenes from BEE.v8i + iNat
crops), so every mask is tightly anchored to an actual insect box. Multiple boxes per
image → multiple instances.

Two more quality levers baked in:
  * **Real-scene data**: uses ``insect_det`` (includes BEE.v8i garden bees in natural
    scenes with flowers/background), not just centred crops → the model learns to
    segment the insect *within* a scene, matching video.
  * **Flower background negatives**: a fraction of pure-flower images are added with
    **empty** label files, teaching the segmenter that flowers are NOT insects — this
    directly fixes the "mask bleeds onto the flower" failure.

Single class ``insect`` (type comes from the species classifier downstream).

Output (git-ignored): data/interim/insect_seg/{images,labels}/{train,val,test} + data.yaml

CLI:  python -m src.cv_engine.sam_to_seg
"""

import random

import cv2
import numpy as np


SRC = C.INTERIM_DIR / "insect_det"           # 2-class boxes: iNat crops + BEE.v8i scenes
FLOWER_SRC = C.INTERIM_DIR / "flower_det"    # flower detection images -> negatives
OUT = C.INTERIM_DIR / "insect_seg"
NEG_FRAC = 0.15                              # flower-only negatives as a fraction of positives


def _mask_to_polygon(mask, W, H, eps_frac=0.004, min_area_frac=0.002):
    """Largest external contour -> normalized YOLO-seg polygon string, or None."""
    cnts, _ = cv2.findContours(mask.astype("uint8"), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < min_area_frac * W * H:
        return None
    ap = cv2.approxPolyDP(c, eps_frac * cv2.arcLength(c, True), True).reshape(-1, 2)
    if len(ap) < 3:
        return None
    return " ".join(f"{x / W:.6f} {y / H:.6f}" for x, y in ap)


def _read_boxes(lbl_path, W, H):
    """Read a YOLO label file -> list of pixel xyxy boxes (class ignored -> insect)."""
    boxes = []
    if not lbl_path.exists():
        return boxes
    for line in lbl_path.read_text().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        cx, cy, w, h = (float(v) for v in p[1:5])
        x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H
        x2, y2 = (cx + w / 2) * W, (cy + h / 2) * H
        boxes.append([max(0, x1), max(0, y1), min(W, x2), min(H, y2)])
    return boxes


def run() -> dict:
    from ultralytics import SAM
    model = SAM("mobile_sam.pt")
    stats = {"labeled": 0, "instances": 0, "skipped": 0, "negatives": 0}
    rng = random.Random(C.SEED)

    for split in ("train", "val", "test"):
        (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)
        img_dir = SRC / "images" / split
        lbl_dir = SRC / "labels" / split
        if not img_dir.is_dir():
            continue
        for p in sorted(img_dir.iterdir()):
            img = cv2.imread(str(p))
            if img is None:
                stats["skipped"] += 1; continue
            H, W = img.shape[:2]
            boxes = _read_boxes(lbl_dir / f"{p.stem}.txt", W, H)
            if not boxes:
                stats["skipped"] += 1; continue
            r = model.predict(img, bboxes=boxes, labels=[1] * len(boxes), verbose=False)[0]
            if r.masks is None or len(r.masks) == 0:
                stats["skipped"] += 1; continue
            lines = []
            for m in r.masks.data.cpu().numpy():
                if m.shape != (H, W):
                    m = cv2.resize(m.astype("uint8"), (W, H), interpolation=cv2.INTER_NEAREST)
                poly = _mask_to_polygon(m > 0, W, H)
                if poly is not None:
                    lines.append(f"0 {poly}")
            if not lines:
                stats["skipped"] += 1; continue
            link = OUT / "images" / split / p.name
            if not link.exists():
                link.symlink_to(p.resolve())
            (OUT / "labels" / split / f"{p.stem}.txt").write_text("\n".join(lines) + "\n")
            stats["labeled"] += 1; stats["instances"] += len(lines)
            if stats["labeled"] % 1000 == 0:
                print(f"labeled {stats['labeled']}  instances {stats['instances']}", flush=True)

        # flower-only background negatives (empty label) -> "flowers are not insects"
        fdir = FLOWER_SRC / "images" / split
        if fdir.is_dir():
            flowers = sorted(fdir.iterdir())
            rng.shuffle(flowers)
            n_neg = int(stats["labeled"] * NEG_FRAC / 3)  # rough per-split share
            for fp in flowers[:n_neg]:
                name = f"neg_{fp.name}"
                link = OUT / "images" / split / name
                if not link.exists():
                    try:
                        link.symlink_to(fp.resolve())
                    except FileNotFoundError:
                        continue
                (OUT / "labels" / split / f"neg_{fp.stem}.txt").write_text("")  # empty = background
                stats["negatives"] += 1

    (OUT / "data.yaml").write_text(
        f"# single-class insect instance segmentation (box-prompted SAM + flower negatives)\n"
        f"path: {OUT.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        f"nc: 1\nnames: [insect]\n")
    return stats


if __name__ == "__main__":
    import json
    print(json.dumps(run(), indent=2))

'''
exec(compile(_src, "src/cv_engine/sam_to_seg.py", "exec"), sam_to_seg.__dict__)
_s.modules["sam_to_seg"] = sam_to_seg
globals()["sam_to_seg"] = sam_to_seg

#### `src/cv_engine/train.py` (inlined)

In [ ]:
# ===== src/cv_engine/train.py  (inlined module — edit here) =====
import types as _t, sys as _s
cvtrain = _t.ModuleType("cvtrain")
cvtrain.C = C
cvtrain.__file__ = str(_REPO / "src" / "cv_engine/train.py")
_src = r'''
"""Fine-tune a YOLO26 detector (flowers or insects) on the RTX 3050 6GB.

Thin, reusable wrapper over Ultralytics. Runs are written under
``data/interim/cv_runs/<name>/`` (git-ignored); best weights at
``.../weights/best.pt``.

CLI:
    python -m src.cv_engine.train --data data/interim/flower_det/data.yaml \
        --name flower_yolo26n --epochs 60 --batch 16
"""

import argparse
import multiprocessing
from pathlib import Path

# Python 3.14 defaults to the "forkserver" start method, which makes Ultralytics'
# DataLoader workers re-import this module and spawn a runaway process swarm with
# no real training. Force "fork" so workers inherit state cleanly.
try:
    multiprocessing.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass  # fork unavailable (Windows) -> use the default start method

from ultralytics import YOLO


RUNS_DIR = C.INTERIM_DIR / "cv_runs"
WEIGHTS_DIR = C.INTERIM_DIR / "weights"


def train(data: str, name: str, model: str = "yolo26n.pt", epochs: int = 60,
          imgsz: int = 640, batch: int = 16, device: int | str = 0,
          patience: int = 15, resume: bool = False, scale: float = 0.5) -> dict:
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    # prefer a local pretrained copy if present (avoids re-download to cwd)
    local = WEIGHTS_DIR / model
    yolo = YOLO(str(local) if local.exists() else model)
    results = yolo.train(
        data=data, epochs=epochs, imgsz=imgsz, batch=batch, device=device,
        project=str(RUNS_DIR), name=name, seed=C.SEED, deterministic=True,
        amp=True, patience=patience, exist_ok=True, resume=resume, verbose=True,
        # scale jitter: larger range makes the detector see objects at more
        # scales (incl. small) -> tighter boxes on small/video insects.
        scale=scale,
    )
    best = RUNS_DIR / name / "weights" / "best.pt"
    # report validation mAP
    metrics = yolo.val(data=data, imgsz=imgsz, device=device, verbose=False)
    out = {
        "name": name, "best_weights": str(best),
        "map50": round(float(metrics.box.map50), 4),
        "map50_95": round(float(metrics.box.map), 4),
    }
    return out


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data", required=True, help="path to data.yaml")
    ap.add_argument("--name", required=True, help="run name")
    ap.add_argument("--model", default="yolo26n.pt")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--patience", type=int, default=15)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--scale", type=float, default=0.5, help="scale-jitter gain (aug)")
    args = ap.parse_args()
    import json
    print(json.dumps(train(args.data, args.name, args.model, args.epochs,
                           args.imgsz, args.batch, patience=args.patience,
                           resume=args.resume, scale=args.scale), indent=2))


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/cv_engine/train.py", "exec"), cvtrain.__dict__)
_s.modules["cvtrain"] = cvtrain
globals()["cvtrain"] = cvtrain

#### `src/cv_engine/visit_counter.py` (inlined)

In [ ]:
# ===== src/cv_engine/visit_counter.py  (inlined module — edit here) =====
import types as _t, sys as _s
visit_counter = _t.ModuleType("visit_counter")
visit_counter.C = C
visit_counter.__file__ = str(_REPO / "src" / "cv_engine/visit_counter.py")
_src = r'''
"""Flower-visit counting on video (two-stage insect recognition).

Pipeline:
  1. Detect flowers on **every frame** and keep stable IDs across frames via IoU
     association (handles moving camera/flowers); each flower's box is dilated
     into an approach ROI (``flower_1, flower_2, ...``).
  2. Detect and track insects across frames with a single-class YOLO26 detector
     + BoT-SORT (one track ID per insect).
  3. Classify each tracked insect crop with the iNaturalist-pretrained classifier
     as ``pollinator`` or ``non_pollinator`` (majority vote over a track's life).
  4. Count a visit whenever a tracked insect enters a flower ROI (enter-transition,
     debounced so one dwell = one visit and tracker flicker does not double-count).

Outputs:
  * ``<video>_visits.csv``  -> flower_id, total, pollinator, non_pollinator
  * annotated ``<video>_annotated.mp4`` (flowers + tracks + live counts) if --save-video

CLI:
    python -m src.cv_engine.visit_counter --video data/raw/Test_Video/clip.mp4 \
        --flower-weights .../flower/best.pt --insect-weights .../insect/best.pt \
        --classifier-weights .../insect_classifier/best.pt --save-video
"""

import argparse
import csv
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np



def _center(b):
    return (b[0] + b[2]) / 2, (b[1] + b[3]) / 2


def _in(box, pt):
    return box[0] <= pt[0] <= box[2] and box[1] <= pt[1] <= box[3]


class Classifier:
    """iNaturalist-pretrained pollinator/non_pollinator classifier (lazy torch)."""

    def __init__(self, weights: str):
        import timm
        import torch
        self.torch = torch
        ckpt = torch.load(weights, map_location="cpu", weights_only=True)
        self.classes = ckpt["classes"]
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = timm.create_model(ckpt["model"], pretrained=False, num_classes=len(self.classes))
        self.model.load_state_dict(ckpt["state_dict"])
        self.model.eval().to(self.device)
        cfg = timm.data.resolve_data_config({}, model=self.model)
        self.tf = timm.data.create_transform(**cfg, is_training=False)

    def predict(self, crop_bgr) -> str:
        from PIL import Image
        if crop_bgr.size == 0:
            return self.classes[0]
        img = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        x = self.tf(img).unsqueeze(0).to(self.device)
        with self.torch.no_grad():
            idx = int(self.model(x).argmax(1).item())
        return self.classes[idx]


def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def _detect_flowers_raw(model, frame, conf, dilate=0.15):
    res = model.predict(frame, conf=conf, verbose=False)[0]
    H, W = frame.shape[:2]
    out = []
    for b in res.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = b
        dw, dh = (x2 - x1) * dilate, (y2 - y1) * dilate
        out.append((max(0, x1 - dw), max(0, y1 - dh), min(W, x2 + dw), min(H, y2 + dh)))
    return out


class FlowerTracker:
    """Per-frame flower detection with stable IDs across frames (IoU association).

    Handles dynamic video (moving camera/flowers): each frame the flowers are
    re-detected and matched to existing tracks by IoU so ``flower_1`` stays the
    same flower as it moves. A short ``max_missed`` grace keeps IDs through brief
    misses. ``seen`` records every flower ID ever assigned (for the final report).
    """

    def __init__(self, model, conf, dilate=0.15, iou_thr=0.3, max_missed=30):
        self.model, self.conf, self.dilate = model, conf, dilate
        self.iou_thr, self.max_missed = iou_thr, max_missed
        self.tracks: dict[str, dict] = {}
        self.next_id = 1
        self.seen: set[str] = set()

    def update(self, frame):
        dets = _detect_flowers_raw(self.model, frame, self.conf, self.dilate)
        used = set()
        for fid in list(self.tracks):
            best, bj = self.iou_thr, -1
            for j, d in enumerate(dets):
                if j in used:
                    continue
                iou = _iou(self.tracks[fid]["box"], d)
                if iou >= best:
                    best, bj = iou, j
            if bj >= 0:
                self.tracks[fid] = {"box": dets[bj], "missed": 0}
                used.add(bj)
            else:
                self.tracks[fid]["missed"] += 1
                if self.tracks[fid]["missed"] > self.max_missed:
                    del self.tracks[fid]
        for j, d in enumerate(dets):
            if j not in used:
                fid = f"flower_{self.next_id}"; self.next_id += 1
                self.tracks[fid] = {"box": d, "missed": 0}
                self.seen.add(fid)
        return self.current()

    def current(self):
        """Active flower boxes without re-detecting (reused between detect frames)."""
        return [(fid, t["box"]) for fid, t in self.tracks.items() if t["missed"] == 0]


POLLINATOR_TYPES = {"bee", "butterfly", "wasp", "fly"}  # highlighted + rolled up in CSV


def _load_types():
    """Map species class_id (str) -> coarse insect type from the taxonomy manifest."""
    import csv as _csv
    bee = set(_C.BEE_FAMILIES)
    out = {}
    try:
        for r in _csv.DictReader(open(_C.MANIFEST_DIR / "split_manifest.csv")):
            o, f = r["order"], r["family"]
            if f in bee: t = "bee"
            elif o == "Lepidoptera": t = "butterfly"
            elif f == "Formicidae": t = "ant"
            elif o == "Coleoptera": t = "beetle"
            elif o == "Diptera": t = "fly"
            elif o == "Hymenoptera": t = "wasp"
            elif o == "Hemiptera": t = "bug"
            else: t = (o or "insect").lower()
            out[r["class_id"]] = t
    except FileNotFoundError:
        pass
    return out


def count_visits(video, flower_weights, insect_weights, classifier_weights,
                 out_dir: Path, conf=0.1, debounce=20, save_video=False,
                 flower_interval=5, vid_stride=2) -> dict:
    from ultralytics import YOLO
    out_dir.mkdir(parents=True, exist_ok=True)
    flower_model, insect_model = YOLO(flower_weights), YOLO(insect_weights)
    classifier = Classifier(classifier_weights) if classifier_weights else None

    types = _load_types() if classifier is not None else {}
    visits = defaultdict(Counter)
    track_votes: dict[int, Counter] = defaultdict(Counter)   # track_id -> class votes
    last_flower: dict[int, str | None] = {}
    last_visit_frame: dict[tuple[int, str], int] = {}
    ftracker = FlowerTracker(flower_model, conf)
    writer = None

    stream = insect_model.track(source=video, stream=True, tracker="botsort.yaml",
                                persist=True, conf=conf, verbose=False, vid_stride=vid_stride)
    for fi, res in enumerate(stream):
        frame = res.orig_img
        # re-detect flowers every `flower_interval` frames (they move slowly);
        # reuse the tracked boxes in between -> dynamic but ~interval× faster.
        flowers = ftracker.update(frame) if fi % flower_interval == 0 else ftracker.current()
        if fi == 0 and save_video:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(str(out_dir / (Path(video).stem + "_annotated.mp4")),
                                     cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
        boxes = res.boxes
        labels: dict[int, str] = {}
        if boxes is not None and boxes.id is not None:
            ids = boxes.id.int().cpu().tolist()
            xyxy = boxes.xyxy.cpu().numpy()
            for tid, b in zip(ids, xyxy):
                if classifier is not None:                    # majority vote per track
                    x1, y1, x2, y2 = map(int, b)
                    sp = classifier.predict(frame[y1:y2, x1:x2])
                    track_votes[tid][types.get(sp, sp)] += 1   # species id -> type
                    cls = track_votes[tid].most_common(1)[0][0]
                else:
                    cls = "insect"
                labels[tid] = cls
                cur = next((fid for fid, fb in flowers if _in(fb, _center(b))), None)
                if cur is not None and last_flower.get(tid) != cur:
                    if fi - last_visit_frame.get((tid, cur), -10**9) > debounce:
                        visits[cur]["total"] += 1
                        visits[cur][cls] += 1
                        last_visit_frame[(tid, cur)] = fi
                last_flower[tid] = cur
        if writer is not None:
            writer.write(_annotate(frame, flowers, res, labels, visits))
    if writer is not None:
        writer.release()

    for fid in ftracker.seen:                     # include flowers seen with 0 visits
        visits[fid]
    ctypes = sorted({t for v in visits.values() for t in v if t != "total"})
    rows = [{"flower_id": k, "total": v["total"],
             "pollinator": sum(v.get(t, 0) for t in POLLINATOR_TYPES),
             "non_pollinator": v["total"] - sum(v.get(t, 0) for t in POLLINATOR_TYPES),
             **{t: v.get(t, 0) for t in ctypes}}
            for k, v in sorted(visits.items())]
    csv_path = out_dir / (Path(video).stem + "_visits.csv")
    with open(csv_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["flower_id", "total", "pollinator", "non_pollinator"] + ctypes)
        w.writeheader(); w.writerows(rows)
    return {"video": Path(video).name, "flowers": len(flowers),
            "visits": {r["flower_id"]: r["total"] for r in rows}, "csv": str(csv_path)}


def _annotate(frame, flowers, res, labels, visits):
    for fid, (x1, y1, x2, y2) in flowers:
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 2)
        cv2.putText(frame, f"{fid}:{visits[fid]['total']}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    if res.boxes is not None and res.boxes.id is not None:
        for tid, b in zip(res.boxes.id.int().cpu().tolist(), res.boxes.xyxy.cpu().numpy()):
            x1, y1, x2, y2 = map(int, b)
            cls = labels.get(tid, "insect")
            poll = cls in POLLINATOR_TYPES
            col = (0, 180, 255) if poll else (0, 0, 255)   # pollinator=orange, else red
            cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
            tag = f"{cls}{' (pollinator)' if poll else ''} #{tid}"
            cv2.putText(frame, tag, (x1, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)
    return frame


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--video", required=True)
    ap.add_argument("--flower-weights", required=True)
    ap.add_argument("--insect-weights", required=True)
    ap.add_argument("--classifier-weights", default="")
    ap.add_argument("--out", default=str(C.INTERIM_DIR / "cv_runs" / "visits"))
    ap.add_argument("--conf", type=float, default=0.1)
    ap.add_argument("--debounce", type=int, default=20)
    ap.add_argument("--flower-interval", type=int, default=5,
                    help="re-detect flowers every N frames (dynamic video)")
    ap.add_argument("--vid-stride", type=int, default=2,
                    help="process every Nth frame (lower effective fps, stabler tracks)")
    ap.add_argument("--save-video", action="store_true")
    args = ap.parse_args()
    import json
    print(json.dumps(count_visits(args.video, args.flower_weights, args.insect_weights,
                                  args.classifier_weights, Path(args.out), args.conf,
                                  args.debounce, args.save_video, args.flower_interval,
                                  args.vid_stride), indent=2))


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/cv_engine/visit_counter.py", "exec"), visit_counter.__dict__)
_s.modules["visit_counter"] = visit_counter
globals()["visit_counter"] = visit_counter

#### `src/cv_engine/video_seg.py` (inlined)

In [ ]:
# ===== src/cv_engine/video_seg.py  (inlined module — edit here) =====
import types as _t, sys as _s
video_seg = _t.ModuleType("video_seg")
video_seg.C = C
video_seg.__file__ = str(_REPO / "src" / "cv_engine/video_seg.py")
for _n in ("Classifier", "FlowerTracker", "POLLINATOR_TYPES", "_in", "_load_types"):
    setattr(video_seg, _n, getattr(visit_counter, _n))
_src = r'''
"""Flower-visit counting on video with **instance-segmented** insects (v2 seg pipeline).

Designed for a single real-time camera stream: the incoming video is resampled to a
fixed **target fps (24)** so tracking is stable and compute is bounded regardless of
the source frame rate.

Per frame:
  1. Flowers: per-frame YOLO **detection** with stable IDs (``FlowerTracker``) -> a
     **separate** box per flower (never unified).
  2. Insects: single-class YOLO26-**seg** + BoT-SORT -> one pixel **mask** + track ID
     each. Masks that cover too much of the frame are dropped (a flower false positive,
     not an insect).
  3. Type: each insect crop -> species classifier -> coarse type (majority vote / track).
  4. Visit: counted when an insect **mask actually overlaps a flower box**
     (mask∩box / mask_area > threshold), as an enter-transition, debounced so a
     fly-off + return is not a second visit. Each visit is stamped with a **timestamp**
     (seconds into the clip).

Rendering uses ``supervision`` so every insect instance gets its **own colour** (by
track ID — bee #1 and bee #2 differ), with the flower boxes drawn on top.

Outputs (to ``test_video_result/`` by default):
  * ``<video>_visits.csv``    -> flower_id, total, <per insect type ...>
  * ``<video>_timeline.csv``  -> flower_id, track_id, type, t_enter_s  (one row per visit)
  * annotated ``<video>_annotated.mp4`` (per-instance masks + flower boxes + counts)

CLI:
    python -m src.cv_engine.video_seg --video data/raw/Test_Video/clip.mp4 \
        --flower-weights .../flower/best.pt --insect-weights .../insect_seg/best.pt \
        --classifier-weights .../species/best.pt --save-video
"""

import argparse
import csv
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np


TARGET_FPS = 24                # single-camera stream is resampled to this
MAX_MASK_FRAC = 0.15           # drop insect masks bigger than this: a whole-flower false
                               # positive covers ~30%+ of the frame; real insects run
                               # ~2-6% even for a large close-up butterfly.
OVERLAP_THR = 0.10             # mask∩flowerbox / mask_area needed to count as "on flower"


def _box_overlap_frac(mask: np.ndarray, box) -> float:
    """Fraction of an insect mask's pixels that fall inside a flower box."""
    a = int(mask.sum())
    if a == 0:
        return 0.0
    x1, y1, x2, y2 = (int(v) for v in box)
    inside = int(mask[max(0, y1):max(0, y2), max(0, x1):max(0, x2)].sum())
    return inside / a


def count_visits_seg(video, flower_weights, insect_weights, classifier_weights,
                     out_dir: Path, conf=0.15, save_video=False,
                     flower_interval=5, target_fps=TARGET_FPS, flower_conf=0.15) -> dict:
    import supervision as sv
    from ultralytics import YOLO
    out_dir.mkdir(parents=True, exist_ok=True)
    flower_model, insect_model = YOLO(flower_weights), YOLO(insect_weights)
    classifier = Classifier(classifier_weights) if classifier_weights else None
    types = _load_types() if classifier is not None else {}

    in_fps = cv2.VideoCapture(str(video)).get(cv2.CAP_PROP_FPS) or 30.0
    stride = max(1, round(in_fps / target_fps))          # resample -> ~target fps
    out_fps = in_fps / stride

    visits = defaultdict(Counter)
    timeline: list[dict] = []
    track_votes: dict[int, Counter] = defaultdict(Counter)
    counted: set[tuple[int, str]] = set()   # (track_id, flower_id) counted once ever
    ftracker = FlowerTracker(flower_model, flower_conf)   # lower conf -> catch more flowers
    mask_ann = sv.MaskAnnotator(color_lookup=sv.ColorLookup.TRACK, opacity=0.55)
    writer = None

    stream = insect_model.track(source=video, stream=True, tracker="botsort.yaml",
                                persist=True, conf=conf, verbose=False, vid_stride=stride)
    for fi, res in enumerate(stream):
        frame = res.orig_img
        H, W = frame.shape[:2]
        t_s = round(fi * stride / in_fps, 2)             # timestamp (s) of this frame
        flowers = ftracker.update(frame) if fi % flower_interval == 0 else ftracker.current()
        if fi == 0 and save_video:
            writer = cv2.VideoWriter(str(out_dir / (Path(video).stem + "_annotated.mp4")),
                                     cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (W, H))

        det = sv.Detections.from_ultralytics(res)
        # keep only tracked insects with a mask that is not flower-sized
        det_draw, labels = sv.Detections.empty(), []
        if det.tracker_id is not None and det.mask is not None and len(det):
            keep = np.zeros(len(det), dtype=bool)
            for k in range(len(det)):
                m = det.mask[k]
                if m.sum() / (H * W) > MAX_MASK_FRAC:     # too big -> flower, not insect
                    continue
                keep[k] = True
                tid = int(det.tracker_id[k])
                if classifier is not None:
                    x1, y1, x2, y2 = (int(v) for v in det.xyxy[k])
                    sp = classifier.predict(frame[y1:y2, x1:x2])
                    track_votes[tid][types.get(sp, sp)] += 1
                    cls = track_votes[tid].most_common(1)[0][0]
                else:
                    cls = "insect"
                # assign to the flower whose box the mask overlaps most
                best_fid, best_ov = None, OVERLAP_THR
                for fid, fb in flowers:
                    ov = _box_overlap_frac(m, fb)
                    if ov >= best_ov:
                        best_ov, best_fid = ov, fid
                # count each (insect, flower) pair only once ever -> a fly-off and
                # return to the same flower is NOT a second visit
                if best_fid is not None and (tid, best_fid) not in counted:
                    counted.add((tid, best_fid))
                    visits[best_fid]["total"] += 1
                    visits[best_fid][cls] += 1
                    timeline.append({"flower_id": best_fid, "track_id": tid,
                                     "type": cls, "t_enter_s": t_s})
            det_draw = det[keep]
            labels = [f"{track_votes[int(t)].most_common(1)[0][0]} #{int(t)}"
                      for t in det_draw.tracker_id]

        if writer is not None:
            annotated = frame.copy()
            if len(det_draw):
                annotated = mask_ann.annotate(annotated, det_draw)
            annotated = _draw_boxes_labels(annotated, det_draw, labels, flowers, visits, sv)
            writer.write(annotated)
    if writer is not None:
        writer.release()

    for fid in ftracker.seen:
        visits[fid]
    ctypes = sorted({t for v in visits.values() for t in v if t != "total"})
    rows = [{"flower_id": k, "total": v["total"], **{t: v.get(t, 0) for t in ctypes}}
            for k, v in sorted(visits.items())]
    csv_path = out_dir / (Path(video).stem + "_visits.csv")
    with open(csv_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["flower_id", "total"] + ctypes)
        w.writeheader(); w.writerows(rows)
    tl_path = out_dir / (Path(video).stem + "_timeline.csv")
    with open(tl_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["flower_id", "track_id", "type", "t_enter_s"])
        w.writeheader(); w.writerows(timeline)
    return {"video": Path(video).name, "flowers": len(ftracker.seen),
            "out_fps": round(out_fps, 1), "visits": {r["flower_id"]: r["total"] for r in rows},
            "csv": str(csv_path), "timeline": str(tl_path)}


def _draw_boxes_labels(frame, det, labels, flowers, visits, sv):
    # per-instance insect label (colour matches the mask, i.e. the track)
    for k in range(len(det)):
        x1, y1 = int(det.xyxy[k][0]), int(det.xyxy[k][1])
        col = sv.ColorPalette.DEFAULT.by_idx(int(det.tracker_id[k])).as_bgr()
        cv2.putText(frame, labels[k], (x1, max(12, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
    # flower boxes on top (separate box per flower) + live count
    for fid, (x1, y1, x2, y2) in flowers:
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 2)
        cv2.putText(frame, f"{fid}:{visits[fid]['total']}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    return frame


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--video", required=True)
    ap.add_argument("--flower-weights", required=True)
    ap.add_argument("--insect-weights", required=True, help="YOLO26-seg insect weights")
    ap.add_argument("--classifier-weights", default="")
    ap.add_argument("--out", default=str(C.REPO_ROOT / "test_video_result"))
    ap.add_argument("--conf", type=float, default=0.25, help="insect seg confidence")
    ap.add_argument("--flower-conf", type=float, default=0.15, help="flower detect confidence")
    ap.add_argument("--flower-interval", type=int, default=5)
    ap.add_argument("--target-fps", type=int, default=TARGET_FPS)
    ap.add_argument("--save-video", action="store_true")
    args = ap.parse_args()
    import json
    print(json.dumps(count_visits_seg(args.video, args.flower_weights, args.insect_weights,
                                      args.classifier_weights, Path(args.out), args.conf,
                                      args.save_video, args.flower_interval,
                                      args.target_fps, args.flower_conf), indent=2))


if __name__ == "__main__":
    main()

'''
exec(compile(_src, "src/cv_engine/video_seg.py", "exec"), video_seg.__dict__)
_s.modules["video_seg"] = video_seg
globals()["video_seg"] = video_seg

## 1 · SAM-bootstrapped segmentation dataset
Turn tight insect boxes into YOLO-seg masks with SAM (idempotent).

In [ ]:
seg_yaml = C.INTERIM_DIR / "insect_seg" / "data.yaml"
if not seg_yaml.exists():
    if not (C.INTERIM_DIR / "insect_det1" / "data.yaml").exists():
        prepare_insect.run(); prepare_insect.export_single_class()
    print(json.dumps(sam_to_seg.run(), indent=2))
else:
    print("seg dataset present:", seg_yaml)
n = {s: len(list((C.INTERIM_DIR / "insect_seg" / "labels" / s).glob("*.txt"))) for s in ("train","val","test")}
print("seg masks:", n)

## 2 · Insect segmenter — `yolo26s-seg`
Idempotent: loads existing `best.pt`.

In [ ]:
seg_run = RUNS / "insect_seg_yolo26s"
seg_w = seg_run / "weights" / "best.pt"
if not seg_w.exists():
    cvtrain.train(str(seg_yaml), "insect_seg_yolo26s", model="yolo26s-seg.pt",
                  epochs=80, imgsz=640, batch=8, patience=15)
flower_w = RUNS / "flower_yolo26n" / "weights" / "best.pt"
clf_w    = RUNS / "species_classifier" / "best.pt"
for label, w in [("segmenter", seg_w), ("flower", flower_w), ("classifier", clf_w)]:
    print(f"{label:10}", w, "|", exists(w))

## 3 · Metrics

In [ ]:
def best_seg_map(csv_path):
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    i = df["metrics/mAP50-95(M)"].idxmax()
    return {k: round(float(df.loc[i, f"metrics/{k}"]), 4)
            for k in ["mAP50(M)", "mAP50-95(M)", "precision(M)", "recall(M)", "mAP50(B)"]}
metrics = {"insect_segmenter": best_seg_map(seg_run / "results.csv"),
           "flower_detector_mAP50(B)": round(pd.read_csv(RUNS / "flower_yolo26n" / "results.csv").rename(columns=str.strip)["metrics/mAP50(B)"].max(), 4)}
print(json.dumps(metrics, indent=2))

## 4 · Video — segment insects, count flower visits
Stream resampled to **24 fps**. Flowers = separate box + ID; insects = **mask +
type + track ID** with a **per-instance colour**; a visit is counted when an insect
mask overlaps a flower box, each (insect, flower) pair **once ever** (fly-off +
return != new visit), stamped with a **timestamp**. Annotated videos + per-flower
CSV + per-visit timeline CSV → root `test_video_result/`.

In [ ]:
out = REPO / "test_video_result"; out.mkdir(exist_ok=True)
results = []
for v in sorted(C.TEST_VIDEO_DIR.glob("*.mp4")):
    r = video_seg.count_visits_seg(str(v), str(flower_w), str(seg_w), str(clf_w),
                                   out, save_video=True, flower_interval=5)
    results.append(r); print(v.name, "->", r["flowers"], "flowers,", sum(r["visits"].values()), "visits @", r["out_fps"], "fps")
print("saved annotated (mask) videos to", out)

### Visit tallies (per flower, per insect type)

In [ ]:
frames = []
for csvf in sorted(out.glob("*_visits.csv")):
    d = pd.read_csv(csvf); d.insert(0, "video", csvf.stem.replace("_visits", ""))
    frames.append(d)
visits_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
visits_df[visits_df["total"] > 0] if not visits_df.empty else visits_df

### Visit timeline (per-visit timestamps)

In [ ]:
tls = []
for tlf in sorted(out.glob("*_timeline.csv")):
    d = pd.read_csv(tlf)
    if len(d):
        d.insert(0, "video", tlf.stem.replace("_timeline", "")); tls.append(d)
timeline_df = pd.concat(tls, ignore_index=True) if tls else pd.DataFrame()
timeline_df.head(20)

### Sample annotated frame (flower boxes + insect masks)

In [ ]:
vids = sorted(glob.glob(str(out / "*_annotated.mp4")))
if vids:
    cap = cv2.VideoCapture(vids[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * 0.5))
    ok, fr = cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(10, 6)); plt.axis("off"); plt.title(Path(vids[0]).stem)
        plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); plt.show()

## 5 · Summary

In [ ]:
print(json.dumps({"metrics": metrics, "videos": len(results),
                  "total_visits": int(sum(sum(r["visits"].values()) for r in results))}, indent=2))